## Scratchwork Week 11/24

In [12]:
%load_ext autoreload
%autoreload 2
import numpy as np 
import sys 
sys.path.append("../discretized-causalpfn")
from discretize import DataDiscretizer
from inference import DiscreteCausalPFN

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
## Synthetic data generation
data_name = "vahid-linear"
np.random.seed(42)
n, d = 2000, 3
X = np.random.normal(1, 1, size=(n, d)).astype(np.float32)
T = (X[:, 0] - X[:, 1] + 2 * X[:, 2] + 2 + np.random.normal(0, 3, size=n)).astype(np.float32)
T = T - T.min() # Rescale
T = T / T.max() # Rescale
Y = (3 * X[:, 0] + X[:, 1] - 0.5 * X[:, 2] + 3 * T + np.random.normal(0, 2, size=n)).astype(np.float32)
def drf(t): return 3.5 + 3 * t # true dose-response function

In [14]:
uniform_discretizer = DataDiscretizer("uniform")
N_DISC = 3
T_discrete, T_vals = uniform_discretizer.discretize_treatment(T, N_DISC)

model = DiscreteCausalPFN(
    comparison_method="all",
    N_DISC=N_DISC,
    device="cpu",
    verbose="True"
)

In [15]:
model.predict_epos(
    X, T_discrete, Y, 
    T_vals, 
    take_mean=True, 
    alpha=0.05
)

Predicting CEPO: 100%|██████████| 3790/3790 [00:15<00:00, 250.01it/s]


({np.float64(0.0): np.float32(4.474863),
  np.float64(0.5): np.float32(4.961786),
  np.float64(1.0): np.float32(5.6156626)},
 {np.float64(0.0): (tensor([4.3082]), tensor([4.6444])),
  np.float64(0.5): (tensor([4.8774]), tensor([5.0457])),
  np.float64(1.0): (tensor([5.4265]), tensor([5.8034]))})

In [16]:
np.array([np.ones(shape=(10000,)), np.ones(shape=(10000,))]).mean(axis=0)

array([1., 1., 1., ..., 1., 1., 1.], shape=(10000,))